# NER — Detección de Entidades en Noticias Económicas Argentinas

**Enfoque híbrido:**
- `spaCy es_core_news_lg` → entidades estándar: **PER** (personas), **ORG** (organizaciones), **LOC** (lugares)
- **Gazetteer curado** → **IND_ECO** (indicadores económicos argentinos)

**Outputs en `resultados_ner/`:**
- `ner_articulos.csv` — entidades por artículo (url, entidad, tipo, n_menciones)
- `ner_oraciones.csv` — entidades por oración (para cruzar con sentimiento BETO)
- `ner_resumen.csv` — ranking global de entidades

In [1]:
# Ejecutar una sola vez si no están instalados
# !pip install spacy
# !python -m spacy download es_core_news_lg

In [2]:
import json
import csv
import re
from pathlib import Path
from collections import Counter, defaultdict

# ── Paths ──
# tesis 3.0 usa estructura plana por tema (ver README raíz): ner_entidades/ y
# noticias/ son hermanas directas, no hace falta bajar por pipeline_productivo/.
BASE_DIR    = Path('../noticias/resultados')
OUTPUT_DIR  = Path('resultados_ner')
OUTPUT_DIR.mkdir(exist_ok=True)

ARCHIVOS_CORPUS = [
    'ambito_economia.json',
    'ambito_finanzas.json',
    'clarin_economia.json',
    'clarin_sitemap.json',
    'eleconomista_sitemap.json',
    'infobae_economia.json',
    'iprofesional_sitemap.json',
    'lanacion_economia.json',
    'tn_economia.json',
]

MIN_LARGO_ORACION   = 30   # ignorar oraciones muy cortas
MAX_LARGO_ORACION   = 600  # truncar muy largas

print('Configuracion lista.')

Configuracion lista.


In [3]:
# Gazetteer: indicadores economicos (subtipo -> lista de terminos, sin canonicalizar:
# cada termino ES el concepto, no hay variantes de una misma "entidad" que fusionar)
# + empresas Merval (canonicalizadas: cada alias apunta a un ticker+sector fijo,
# asi "Banco Galicia" / "GGAL" / "Grupo Financiero Galicia" cuentan como UNA sola
# entidad en el output, no tres -- ver AUDITORIA hallazgo de fragmentacion).

GAZETTEER_IND_ECO = {
    'macro': [
        'inflacion', 'inflacion anual', 'inflacion mensual', 'inflacion interanual',
        'inflacion acumulada', 'nucleo inflacionario', 'indice de precios',
        'PBI', 'PIB', 'producto bruto interno', 'crecimiento economico', 'recesion',
        'estanflacion', 'estagflacion',
        'tipo de cambio', 'dolar blue', 'dolar oficial', 'dolar mep', 'dolar bolsa',
        'contado con liquidacion', 'contado con liqui', 'CCL', 'dolar libre',
        'dolar financiero', 'dolar cripto', 'dolar tarjeta', 'dolar ahorro',
        'cepo cambiario', 'cepo al dolar', 'brecha cambiaria', 'devaluacion',
        'apreciacion cambiaria', 'depreciacion', 'crawling peg', 'banda cambiaria',
        'reservas internacionales', 'reservas brutas', 'reservas netas',
        'deuda externa', 'deuda publica', 'deuda soberana', 'restructuracion de deuda',
        'default', 'cesacion de pagos',
        'deficit fiscal', 'superavit fiscal', 'deficit primario', 'resultado primario',
        'equilibrio fiscal', 'ajuste fiscal',
        'balanza comercial', 'exportaciones', 'importaciones', 'saldo comercial',
        'terminos de intercambio', 'cuenta corriente',
    ],
    'monetario': [
        'tasa de interes', 'tasa de politica monetaria', 'tasa de referencia',
        'tasa activa', 'tasa pasiva', 'tasa de descuento',
        'LELIQ', 'Notaliq', 'pases pasivos', 'pases activos',
        'base monetaria', 'emision monetaria', 'expansion monetaria',
        'encajes bancarios', 'credito bancario', 'prestamos bancarios',
        'politica monetaria', 'agregados monetarios',
    ],
    'fiscal': [
        'recaudacion tributaria', 'recaudacion impositiva',
        'gasto publico', 'gasto primario', 'presupuesto nacional',
        'retenciones', 'derechos de exportacion',
        'subsidios energeticos', 'subsidios al transporte', 'subsidios',
        'tarifas electricas', 'tarifa de gas', 'tarifas de servicios',
        'reforma tributaria', 'blanqueo', 'moratoria',
        'impuesto a las ganancias', 'bienes personales', 'IVA', 'ingresos brutos',
    ],
    'financiero': [
        'Merval', 'indice Merval', 'bolsa de valores', 'mercado de capitales',
        'riesgo pais', 'EMBI', 'spread soberano',
        'bonos soberanos', 'bonos en dolares', 'bonos en pesos', 'bono dual',
        'acciones argentinas', 'ADR', 'CEDEAR',
        'fondos comunes de inversion', 'FCI',
        'plazo fijo', 'plazo fijo UVA',
        'tasa de rendimiento', 'yield', 'paridad de bonos',
        'licitacion de deuda', 'mercado primario', 'mercado secundario',
    ],
    'laboral': [
        'tasa de desempleo', 'tasa de desocupacion', 'desempleo',
        'empleo formal', 'empleo informal', 'empleo en negro', 'trabajo no registrado',
        'salario real', 'salario nominal', 'poder adquisitivo',
        'salario minimo vital y movil', 'SMVM',
        'paritarias', 'negociacion paritaria', 'acuerdo salarial',
        'convenio colectivo de trabajo', 'CCT',
        'jubilaciones', 'haberes previsionales', 'movilidad jubilatoria',
        'UVA', 'CER', 'ajuste por inflacion',
    ],
    'precios': [
        'IPC', 'indice de precios al consumidor',
        'canasta basica alimentaria', 'canasta basica total', 'CBA', 'CBT',
        'tasa de pobreza', 'tasa de indigencia', 'pobreza', 'indigencia',
        'precios relativos', 'precios regulados', 'precios maximos',
        'desaceleracion de precios', 'desinflacion', 'deflacion',
        'precio de la soja', 'precio del petroleo', 'commodities',
    ],
}

# Panel Merval canonicalizado: ticker + sector + alias de superficie.
# 'entidad' final en el output = ticker (ej. 'GGAL'), 'subtipo' final = sector.
#
# Criterio de alias (auditoria de precision sobre el corpus, ver README.md):
#  - Los alias en MAYUSCULAS (tickers/siglas) se buscan respetando mayusculas: en texto libre
#    "agro", "loma", "come", "long", "semi" son palabras comunes (con la busqueda
#    insensible, "AGRO" matcheaba la palabra "agro" en 920 articulos).
#  - Se omiten alias genericos que son palabras comunes: "Hipotecario" (credito hipotecario),
#    "Celulosa", "Molinos" (molinos harineros), "Capex" (gasto de capital).
MERVAL_EMPRESAS = [
    {'ticker': 'YPFD',  'sector': 'Energia',       'aliases': ['YPF', 'Yacimientos Petroliferos Fiscales', 'YPFD']},
    {'ticker': 'GGAL',  'sector': 'Financiero',     'aliases': ['Banco Galicia', 'Grupo Financiero Galicia', 'GGAL']},
    {'ticker': 'PAMP',  'sector': 'Utilities',      'aliases': ['Pampa Energia', 'Pampa Energía', 'PAMP']},
    {'ticker': 'BMA',   'sector': 'Financiero',     'aliases': ['Banco Macro', 'BMA']},
    {'ticker': 'TXAR',  'sector': 'Materiales',     'aliases': ['Ternium Argentina', 'Ternium', 'Siderar', 'TXAR']},
    {'ticker': 'CEPU',  'sector': 'Utilities',      'aliases': ['Central Puerto', 'CEPU']},
    {'ticker': 'BBAR',  'sector': 'Financiero',     'aliases': ['BBVA Argentina', 'BBVA', 'Banco Frances', 'Banco Francés', 'BBVA Banco Frances', 'BBAR']},
    {'ticker': 'BYMA',  'sector': 'Financiero',     'aliases': ['BYMA', 'Bolsas y Mercados Argentinos']},
    {'ticker': 'COME',  'sector': 'Conglomerado',       'aliases': ['Comercial del Plata', 'Sociedad Comercial del Plata', 'COME']},
    {'ticker': 'TGSU2', 'sector': 'Energia',        'aliases': ['TGS', 'Transportadora de Gas del Sur', 'TGSU2']},
    {'ticker': 'LOMA',  'sector': 'Materiales',     'aliases': ['Loma Negra', 'LOMA']},
    {'ticker': 'ALUA',  'sector': 'Materiales',     'aliases': ['Aluar', 'Aluar Aluminio', 'ALUA']},
    {'ticker': 'TGNO4', 'sector': 'Energia',        'aliases': ['TGN', 'Transportadora de Gas del Norte', 'TGNO4']},
    {'ticker': 'CRES',  'sector': 'Conglomerado',    'aliases': ['Cresud', 'CRES']},
    {'ticker': 'SUPV',  'sector': 'Financiero',     'aliases': ['Banco Supervielle', 'Grupo Supervielle', 'Supervielle', 'SUPV']},
    {'ticker': 'TECO2', 'sector': 'Comunicaciones', 'aliases': ['Telecom Argentina', 'TECO2']},
    {'ticker': 'MIRG',  'sector': 'Consumo',        'aliases': ['Mirgor', 'MIRG']},
    {'ticker': 'EDN',   'sector': 'Utilities',      'aliases': ['Edenor', 'EDN']},
    {'ticker': 'TRAN',  'sector': 'Utilities',      'aliases': ['Transener', 'TRAN']},
    {'ticker': 'CVH',   'sector': 'Comunicaciones', 'aliases': ['Cablevision Holding', 'Cablevisión Holding', 'Cablevision', 'CVH']},
    {'ticker': 'VALO',  'sector': 'Financiero',     'aliases': ['Banco de Valores', 'Grupo Financiero Valores', 'VALO']},
    # ── Panel ampliado 2026-09-13 (27 tickers nuevos, ver README.md) ──
    {'ticker': 'GARO',  'sector': 'Conglomerado',   'aliases': ['Garovaglio y Zorraquin', 'Garovaglio', 'GARO']},
    # AGRO.BA = Agrometal S.A.I. (verificado: la lista recibida decia "AgroIndustria", error de rotulo).
    # Sin alias "AGRO": en el corpus es la palabra "agro" (sector) y el unico "AGRO" en mayusculas es
    # Adecoagro (NYSE: AGRO), otra empresa. Sector Industrial segun la lista vigente.
    {'ticker': 'AGRO',  'sector': 'Industrial',     'aliases': ['Agrometal']},
    {'ticker': 'GRIM',  'sector': 'Consumo',        'aliases': ['Grimoldi', 'GRIM']},
    {'ticker': 'HAVA',  'sector': 'Consumo',        'aliases': ['Havanna Holding', 'Havanna', 'HAVA']},
    {'ticker': 'LEDE',  'sector': 'Consumo',        'aliases': ['Ledesma', 'LEDE']},
    {'ticker': 'LONG',  'sector': 'Consumo',        'aliases': ['Longvie', 'LONG']},
    {'ticker': 'MOLI',  'sector': 'Consumo',        'aliases': ['Molinos Rio de la Plata', 'MOLI']},
    {'ticker': 'MORI',  'sector': 'Consumo',        'aliases': ['Morixe Hermanos', 'Morixe', 'MORI']},
    # San Miguel (SAMI): el nombre es generico (ciudad/santo) -- solo el ticker
    {'ticker': 'SAMI',  'sector': 'Consumo',        'aliases': ['SAMI']},
    {'ticker': 'SEMI',  'sector': 'Consumo',        'aliases': ['Molinos Juan Semino', 'Juan Semino', 'SEMI']},
    {'ticker': 'CAPX',  'sector': 'Energia',        'aliases': ['CAPEX SA', 'CAPX']},
    {'ticker': 'OEST',  'sector': 'Industrial',    'aliases': ['Grupo Concesionario del Oeste', 'Concesionario del Oeste', 'OEST']},
    {'ticker': 'BHIP',  'sector': 'Financiero',     'aliases': ['Banco Hipotecario', 'BHIP']},
    {'ticker': 'BPAT',  'sector': 'Financiero',     'aliases': ['Banco Patagonia', 'BPAT']},
    # GCDI S.A. (ex TGLT): el nombre actual es generico -- ticker y el nombre anterior
    {'ticker': 'GCDI',  'sector': 'Real Estate',    'aliases': ['GCDI', 'TGLT']},
    {'ticker': 'AUSO',  'sector': 'Industrial',     'aliases': ['Autopistas del Sol', 'AUSO']},
    {'ticker': 'BOLT',  'sector': 'Tecnologia',    'aliases': ['Boldt', 'BOLT']},
    {'ticker': 'CELU',  'sector': 'Materiales',     'aliases': ['Celulosa Argentina', 'CELU']},
    {'ticker': 'FIPL',  'sector': 'Materiales',     'aliases': ['Fiplasto', 'FIPL']},
    {'ticker': 'CTIO',  'sector': 'Real Estate',    'aliases': ['Consultatio', 'CTIO']},
    {'ticker': 'IRSA',  'sector': 'Real Estate',    'aliases': ['IRSA', 'IRSA Inversiones y Representaciones']},
    {'ticker': 'CGPA2', 'sector': 'Utilities',      'aliases': ['Camuzzi Gas Pampeana', 'Camuzzi', 'CGPA2']},
    {'ticker': 'DGCU2', 'sector': 'Utilities',      'aliases': ['Distribuidora de Gas Cuyana', 'DGCU2']},
    {'ticker': 'GBAN',  'sector': 'Utilities',      'aliases': ['Naturgy BAN', 'Gas Natural BAN', 'GBAN']},
    {'ticker': 'METR',  'sector': 'Utilities',      'aliases': ['MetroGas', 'Metrogas', 'METR']},
    {'ticker': 'RIGO',  'sector': 'Materiales',     'aliases': ['Rigolleau', 'RIGO']},
]

# Alias que el texto escribe como nombre propio ("Irsa", "Byma") y por eso se
# buscan SIN respetar mayusculas, aunque esten escritos en mayusculas.
ALIAS_CASE_INSENSITIVE = {'IRSA', 'BYMA'}

# Alias genericos excluidos del matching por texto libre.
ALIAS_AMBIGUOS = {'san miguel'}

# Compilar patrones regex (case-insensitive, ignorar tildes en match)
import unicodedata

def normalizar(texto):
    """Minusculas y sin tildes para comparacion."""
    texto = texto.lower()
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

# GAZETTEER_PATTERNS: se busca sobre el texto normalizado (sin tildes, minusculas).
# GAZETTEER_PATTERNS_CS: alias en mayusculas (tickers/siglas), se busca sobre el texto
# ORIGINAL respetando mayusculas.
# Ambos: {patron: (termino_o_ticker, tipo, subtipo)}
GAZETTEER_PATTERNS = {}
GAZETTEER_PATTERNS_CS = {}

for subtipo, terminos in GAZETTEER_IND_ECO.items():
    for termino in terminos:
        patron = re.compile(r'\b' + re.escape(normalizar(termino)) + r'\b')
        GAZETTEER_PATTERNS[patron] = (termino, 'IND_ECO', subtipo)

for empresa in MERVAL_EMPRESAS:
    ticker = empresa['ticker']
    sector = empresa['sector']
    for alias in empresa['aliases']:
        if normalizar(alias) in ALIAS_AMBIGUOS:
            continue
        if alias.isupper() and alias not in ALIAS_CASE_INSENSITIVE:
            GAZETTEER_PATTERNS_CS[re.compile(r'\b' + re.escape(alias) + r'\b')] = (ticker, 'EMP_MERVAL', sector)
        else:
            GAZETTEER_PATTERNS[re.compile(r'\b' + re.escape(normalizar(alias)) + r'\b')] = (ticker, 'EMP_MERVAL', sector)

# Todo lo que el gazetteer ya cubre (normalizado): las entidades de spaCy con ese
# mismo texto se descartan en el post-procesamiento para no contar dos veces.
TERMINOS_GAZETTEER_NORM = (
    {normalizar(t) for ts in GAZETTEER_IND_ECO.values() for t in ts}
    | {normalizar(a) for e in MERVAL_EMPRESAS for a in e['aliases']}
)

n_ind_eco = sum(1 for v in GAZETTEER_PATTERNS.values() if v[1] == 'IND_ECO')
n_merval  = (sum(1 for v in GAZETTEER_PATTERNS.values() if v[1] == 'EMP_MERVAL')
             + len(GAZETTEER_PATTERNS_CS))
print(f'Gazetteer: {n_ind_eco + n_merval} patrones  ({n_ind_eco} IND_ECO | {n_merval} EMP_MERVAL '
      f'[{len(GAZETTEER_PATTERNS_CS)} solo-mayusculas], {len(MERVAL_EMPRESAS)} tickers)')


Gazetteer: 274 patrones  (160 IND_ECO | 114 EMP_MERVAL [50 solo-mayusculas], 47 tickers)


In [4]:
import spacy

print('Cargando modelo spaCy es_core_news_lg...')
nlp = spacy.load('es_core_news_lg')

# Desactivar lo que no necesitamos (mantenemos: tok2vec, senter, ner)
pipes_a_desactivar = ['morphologizer', 'parser', 'attribute_ruler', 'lemmatizer']
pipes_a_desactivar = [p for p in pipes_a_desactivar if p in nlp.pipe_names]
nlp.disable_pipes(pipes_a_desactivar)

# Asegurar que senter este activo
if 'senter' in nlp.component_names and 'senter' not in nlp.pipe_names:
    nlp.enable_pipe('senter')

print(f'Modelo cargado. Pipes activos: {nlp.pipe_names}')


Cargando modelo spaCy es_core_news_lg...


Modelo cargado. Pipes activos: ['tok2vec', 'senter', 'ner']


In [5]:
# ── Sentence splitter (mismo criterio que BETO) ──
def split_oraciones(texto):
    try:
        import nltk
        try:
            tokenizer = nltk.data.load('tokenizers/punkt_tab/Spanish.pickle')
        except Exception:
            try:
                tokenizer = nltk.data.load('tokenizers/punkt/Spanish.pickle')
            except Exception:
                nltk.download('punkt', quiet=True)
                nltk.download('punkt_tab', quiet=True)
                tokenizer = nltk.data.load('tokenizers/punkt/Spanish.pickle')
        oraciones = tokenizer.tokenize(texto)
    except Exception:
        oraciones = re.split(r'(?<=[.!?])\s+(?=[A-ZAEIOUA-ZAEIOU"])', texto)
    resultado = []
    for o in oraciones:
        o = o.strip().replace('\n', ' ')
        o = re.sub(r'\s+', ' ', o)
        if MIN_LARGO_ORACION <= len(o) <= MAX_LARGO_ORACION:
            resultado.append(o)
    return resultado


# ── NER con spaCy ──
TIPOS_SPACY = {'PER', 'ORG', 'LOC', 'MISC'}

def ner_spacy(texto):
    """Retorna lista de (entidad_texto, tipo)."""
    doc = nlp(texto[:100000])  # spaCy tiene limite interno
    entidades = []
    for ent in doc.ents:
        if ent.label_ in TIPOS_SPACY:
            texto_ent = ent.text.strip()
            if len(texto_ent) > 1:
                entidades.append((texto_ent, ent.label_, None))
    return entidades


# ── NER con gazetteer ──
def ner_gazetteer(texto):
    """
    Retorna lista de (termino_o_ticker, tipo, subtipo) sin repetir la misma entidad
    dentro de la oracion (una empresa nombrada como "Grupo Financiero Galicia (GGAL)"
    cuenta una vez, no dos). Los alias en mayusculas (tickers/siglas) se buscan sobre
    el texto original respetando mayusculas; el resto sobre el texto normalizado.
    """
    texto_norm = normalizar(texto)
    entidades = []
    vistos = set()
    for patron, ent in GAZETTEER_PATTERNS.items():
        if ent not in vistos and patron.search(texto_norm):
            vistos.add(ent)
            entidades.append(ent)
    for patron, ent in GAZETTEER_PATTERNS_CS.items():
        if ent not in vistos and patron.search(texto):
            vistos.add(ent)
            entidades.append(ent)
    return entidades


# ── Procesar un articulo completo ──
def procesar_articulo(articulo):
    """
    Retorna:
      - entidades_articulo: lista de (entidad, tipo, subtipo, n_menciones)
      - entidades_oraciones: lista de dicts con oracion_idx, oracion, entidad, tipo, subtipo
    """
    contenido = articulo.get('contenido', '')
    titulo = articulo.get('titulo', '')
    texto_completo = (titulo + '. ' + contenido).strip()

    oraciones = split_oraciones(texto_completo)

    # Acumuladores a nivel articulo
    conteo_articulo = defaultdict(int)  # (entidad_norm, tipo, subtipo) -> count
    registros_oraciones = []

    for idx, oracion in enumerate(oraciones):
        ents = ner_spacy(oracion) + ner_gazetteer(oracion)
        vistas = set()
        for (entidad, tipo, subtipo) in ents:
            key = (entidad.lower(), tipo, subtipo)
            conteo_articulo[key] += 1
            if key not in vistas:
                registros_oraciones.append({
                    'oracion_idx': idx,
                    'oracion': oracion,
                    'entidad': entidad,
                    'tipo': tipo,
                    'subtipo': subtipo or '',
                })
                vistas.add(key)

    # Construir lista de entidades del articulo
    entidades_articulo = []
    for (entidad_norm, tipo, subtipo), count in conteo_articulo.items():
        entidades_articulo.append({
            'entidad': entidad_norm,
            'tipo': tipo,
            'subtipo': subtipo or '',
            'n_menciones': count,
        })

    return entidades_articulo, registros_oraciones


print('Funciones NER listas.')

Funciones NER listas.


In [6]:
def cargar_corpus():
    articulos = []
    for archivo in ARCHIVOS_CORPUS:
        ruta = BASE_DIR / archivo
        if not ruta.exists():
            print(f'  Advertencia: no encontrado -> {ruta}')
            continue
        with open(ruta, 'r', encoding='utf-8') as f:
            data = json.load(f)
        noticias = data.get('noticias', data) if isinstance(data, dict) else data
        fuente = archivo.replace('.json', '')
        cargados = 0
        for n in noticias:
            if n.get('contenido') and len(n['contenido']) > MIN_LARGO_ORACION:
                n['_fuente'] = fuente
                n['_fecha'] = (n.get('fecha_publicacion') or '')[:10]
                articulos.append(n)
                cargados += 1
        print(f'  {archivo}: {cargados} articulos')
    print(f'\nTotal: {len(articulos)} articulos')
    return articulos

articulos = cargar_corpus()

  ambito_economia.json: 498 articulos
  ambito_finanzas.json: 240 articulos
  clarin_economia.json: 294 articulos
  clarin_sitemap.json: 3737 articulos
  eleconomista_sitemap.json: 2695 articulos
  infobae_economia.json: 500 articulos


  iprofesional_sitemap.json: 5455 articulos
  lanacion_economia.json: 491 articulos
  tn_economia.json: 102 articulos

Total: 14012 articulos


In [7]:
import time
from tqdm.auto import tqdm

print(f'Procesando {len(articulos)} articulos (spaCy en batch, no oracion por oracion)...\n')

# Paso 1: metadata + split de oraciones + gazetteer por oracion (rapido, no
# necesita spaCy -- se hace antes para poder mandar TODAS las oraciones del
# corpus a nlp.pipe() de una, en vez de llamar nlp(oracion) una por una.
# (probado: nlp.pipe con batch_size grande es ~1.5x mas rapido en este modelo;
#  n_process>1 se probo tambien pero para oraciones cortas el overhead de
#  IPC entre procesos supera la ganancia -- quedo descartado, ver README.)
articulo_meta = []
oraciones_planas = []
oracion_articulo_idx = []
oracion_idx_local = []
gazetteer_por_oracion = []

for i, art in enumerate(tqdm(articulos, desc='1/2 split + gazetteer', unit='art')):
    url    = art.get('url', f'art_{i}')
    fuente = art.get('_fuente', '')
    fecha  = art.get('_fecha', '')
    titulo = art.get('titulo', '')
    articulo_meta.append({'url': url, 'fuente': fuente, 'fecha': fecha, 'titulo': titulo})

    contenido = art.get('contenido', '')
    texto_completo = (titulo + '. ' + contenido).strip()
    oraciones = split_oraciones(texto_completo)

    for idx, oracion in enumerate(oraciones):
        oraciones_planas.append(oracion)
        oracion_articulo_idx.append(i)
        oracion_idx_local.append(idx)
        gazetteer_por_oracion.append(ner_gazetteer(oracion))

print(f'\nTotal de oraciones a procesar con spaCy: {len(oraciones_planas)}')

# Paso 2: spaCy en batch sobre TODO el corpus de oraciones de una sola vez.
t_spacy = time.time()
spacy_por_oracion = [None] * len(oraciones_planas)
for j, doc in enumerate(tqdm(
        nlp.pipe(oraciones_planas, batch_size=500),
        total=len(oraciones_planas), desc='2/2 spaCy NER', unit='or')):
    ents = []
    for ent in doc.ents:
        if ent.label_ in TIPOS_SPACY:
            texto_ent = ent.text.strip()
            if len(texto_ent) > 1:
                ents.append((texto_ent, ent.label_, None))
    spacy_por_oracion[j] = ents
print(f'  spaCy listo en {time.time()-t_spacy:.1f}s')

# Paso 3: reensamblar por articulo (misma logica de acumulacion que
# procesar_articulo() antes, solo que ya con las entidades resueltas).
rows_articulos = []
rows_oraciones = []

conteo_por_articulo = defaultdict(lambda: defaultdict(int))  # art_idx -> (entidad_norm,tipo,subtipo) -> count

for j in range(len(oraciones_planas)):
    art_idx   = oracion_articulo_idx[j]
    oracion   = oraciones_planas[j]
    idx_local = oracion_idx_local[j]
    ents = spacy_por_oracion[j] + gazetteer_por_oracion[j]

    vistas_oracion = set()
    for (entidad, tipo, subtipo) in ents:
        key = (entidad.lower(), tipo, subtipo)
        conteo_por_articulo[art_idx][key] += 1
        if key not in vistas_oracion:
            meta = articulo_meta[art_idx]
            rows_oraciones.append({
                'url': meta['url'], 'fuente': meta['fuente'], 'fecha': meta['fecha'], 'titulo': meta['titulo'],
                'oracion_idx': idx_local, 'oracion': oracion,
                'entidad': entidad, 'tipo': tipo, 'subtipo': subtipo or '',
            })
            vistas_oracion.add(key)

for art_idx, conteo in conteo_por_articulo.items():
    meta = articulo_meta[art_idx]
    for (entidad_norm, tipo, subtipo), count in conteo.items():
        rows_articulos.append({
            'url': meta['url'], 'fuente': meta['fuente'], 'fecha': meta['fecha'], 'titulo': meta['titulo'],
            'entidad': entidad_norm, 'tipo': tipo, 'subtipo': subtipo or '', 'n_menciones': count,
        })

print(f'\nListo.')
print(f'  Filas ner_articulos: {len(rows_articulos)}')
print(f'  Filas ner_oraciones: {len(rows_oraciones)}')

C:\Proyectos\Tesis\tesis 4.0\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Procesando 14012 articulos (spaCy en batch, no oracion por oracion)...



1/2 split + gazetteer:   0%|          | 0/14012 [00:00<?, ?art/s]

1/2 split + gazetteer:   0%|          | 1/14012 [00:00<2:45:50,  1.41art/s]

1/2 split + gazetteer:   0%|          | 5/14012 [00:00<30:31,  7.65art/s]  

...

2/2 spaCy NER: 100%|█████████▉| 401501/402352 [20:10<00:02, 375.87or/s]

2/2 spaCy NER: 100%|█████████▉| 402001/402352 [20:11<00:00, 414.36or/s]

2/2 spaCy NER: 100%|██████████| 402352/402352 [20:11<00:00, 332.22or/s]

  spaCy listo en 1211.1s



Listo.
  Filas ner_articulos: 386187
  Filas ner_oraciones: 568234


In [8]:
# ── Post-procesamiento: limpieza de ruido de spaCy ──
# El gazetteer (IND_ECO / EMP_MERVAL) es curado a mano y NO se filtra aca --
# se confia en el aunque una entidad aparezca en un solo articulo.
# Las entidades de spaCy (PER/ORG/LOC/MISC) si necesitan limpieza, ver
# README.md: 65.352 entidades unicas en la corrida anterior, 67.5% hapax,
# mayormente ruido ("75%/4%", fragmentos de oracion) + contaminacion de
# contenido ("PUBLICIDAD" como entidad #1 en menciones).

BLOCKLIST_ENTIDADES = {
    'publicidad',
    # relleno del scraping de Clarin ("Sobre la firma / Mira tambien / Newsletter Clarin"), 2026-10-18
    'newsletter',
    'mira',
}

# Autorreferencias: el nombre del propio medio en sus notas (firma, "segui leyendo en ...").
# Solo se descarta en las notas de ESE medio; si otro medio lo cita, es una mencion legitima.
# La Nacion: solo la marca sin tilde ("la nacion"); "la Nación" con tilde puede ser el Estado nacional.
AUTORREFERENCIAS = {
    'ambito_economia': {'ambito', 'ambito financiero'},
    'ambito_finanzas': {'ambito', 'ambito financiero'},
    'clarin_economia': {'clarin'},
    'clarin_sitemap': {'clarin'},
    'eleconomista_sitemap': {'el economista'},
    'infobae_economia': {'infobae'},
    'iprofesional_sitemap': {'iprofesional'},
    'tn_economia': {'tn', 'todo noticias'},
}
AUTORREFERENCIAS_EXACTAS = {'lanacion_economia': {'la nacion'}}   # sin normalizar tildes


def es_autorreferencia(r):
    fuente = r.get('fuente', '')
    if normalizar(r['entidad']) in AUTORREFERENCIAS.get(fuente, set()):
        return True
    return r['entidad'].strip().lower() in AUTORREFERENCIAS_EXACTAS.get(fuente, set())

MIN_ARTICULOS_SPACY = 2  # frecuencia minima (en articulos distintos) para entidades de spaCy

TIPOS_GAZETTEER = ('IND_ECO', 'EMP_MERVAL')


def es_forma_valida(texto):
    """Heuristica: rechaza fragmentos que casi seguro no son una entidad real."""
    t = texto.strip()
    if len(t) < 2:
        return False
    if re.fullmatch(r'[\d\s.,%\-–—/()"\']+', t):
        return False  # solo numeros/simbolos, ej. "75%/4%"
    if len(t.split()) > 6:
        return False  # fragmento de oracion, no una entidad
    return True


def tiene_palabra_bloqueada(entidad_norm):
    """
    Chequea PALABRA COMPLETA dentro de la entidad, no solo match exacto:
    "PUBLICIDAD" aparece pegada a otras palabras en el HTML original
    ("publicidad a", "bloomberg publicidad", etc.) porque el bloque de
    anuncio queda embebido a mitad de oracion -- un match exacto contra
    "publicidad" a secas no alcanza a cazar esas variantes.
    """
    palabras = set(entidad_norm.split())
    return bool(palabras & BLOCKLIST_ENTIDADES)


def post_procesar(rows_articulos, rows_oraciones):
    # 1) Tipo mayoritario por entidad de spaCy (la misma cadena puede haber
    #    salido como LOC en un articulo y ORG en otro -- se unifica al tipo
    #    con mas menciones totales). rows_articulos ya trae 'entidad' en
    #    minuscula (viene de conteo_articulo); rows_oraciones trae el texto
    #    tal cual lo detecto spaCy (case original) -- se busca por .lower()
    #    en ambos casos para que la reconciliacion cubra los dos.
    menciones = defaultdict(int)
    for r in rows_articulos:
        if r['tipo'] not in TIPOS_GAZETTEER:
            menciones[(r['entidad'], r['tipo'])] += r['n_menciones']

    tipo_final = {}
    for (entidad, tipo), n in menciones.items():
        mejor = tipo_final.get(entidad)
        if mejor is None or n > menciones[(entidad, mejor)]:
            tipo_final[entidad] = tipo

    def remap(r):
        if r['tipo'] not in TIPOS_GAZETTEER:
            clave = r['entidad'].lower()
            if clave in tipo_final:
                r['tipo'] = tipo_final[clave]
        return r

    rows_articulos = [remap(r) for r in rows_articulos]
    rows_oraciones  = [remap(r) for r in rows_oraciones]

    # 2) Re-agregar rows_articulos por (url, entidad, tipo): la reconciliacion
    #    de tipo puede fusionar dos filas del mismo articulo en una.
    agregado = {}
    for r in rows_articulos:
        key = (r['url'], r['entidad'], r['tipo'])
        if key not in agregado:
            agregado[key] = dict(r)
        else:
            agregado[key]['n_menciones'] += r['n_menciones']
    rows_articulos = list(agregado.values())

    # 3) Blocklist (por palabra, no match exacto) + forma valida + duplicado del gazetteer.
    #    Solo entidades de spaCy; gazetteer intacto. Si spaCy detecta "YPF" o "CCL" y el
    #    gazetteer ya cubre ese mismo texto, la fila de spaCy se descarta: sino la misma
    #    empresa/indicador figuraba dos veces (ORG "ypf" + EMP_MERVAL "ypfd").
    def pasa_calidad(r):
        if r['tipo'] in TIPOS_GAZETTEER:
            return True
        ent_norm = normalizar(r['entidad'])
        if ent_norm in TERMINOS_GAZETTEER_NORM:
            return False
        if tiene_palabra_bloqueada(ent_norm):
            return False
        if es_autorreferencia(r):
            return False
        return es_forma_valida(r['entidad'])

    rows_articulos = [r for r in rows_articulos if pasa_calidad(r)]
    rows_oraciones  = [r for r in rows_oraciones if pasa_calidad(r)]

    # 4) Frecuencia minima (solo entidades de spaCy): en cuantos articulos
    #    DISTINTOS aparece la entidad (comparando en minuscula), no cuantas
    #    veces en total.
    articulos_por_entidad = defaultdict(set)
    for r in rows_articulos:
        if r['tipo'] not in TIPOS_GAZETTEER:
            articulos_por_entidad[r['entidad'].lower()].add(r['url'])

    entidades_frecuentes = {
        e for e, urls in articulos_por_entidad.items()
        if len(urls) >= MIN_ARTICULOS_SPACY
    }

    def pasa_frecuencia(r):
        return r['tipo'] in TIPOS_GAZETTEER or r['entidad'].lower() in entidades_frecuentes

    n_antes = len(rows_articulos)
    rows_articulos = [r for r in rows_articulos if pasa_frecuencia(r)]
    rows_oraciones  = [r for r in rows_oraciones if pasa_frecuencia(r)]

    print(f'Post-procesamiento: {n_antes} -> {len(rows_articulos)} filas en ner_articulos '
          f'(tipo reconciliado + blocklist + forma valida + duplicado del gazetteer + frecuencia >= {MIN_ARTICULOS_SPACY} articulos)')

    return rows_articulos, rows_oraciones


rows_articulos, rows_oraciones = post_procesar(rows_articulos, rows_oraciones)
print(f'Filas finales -> ner_articulos: {len(rows_articulos)} | ner_oraciones: {len(rows_oraciones)}')

Post-procesamiento: 351112 -> 312785 filas en ner_articulos (tipo reconciliado + blocklist + forma valida + duplicado del gazetteer + frecuencia >= 2 articulos)
Filas finales -> ner_articulos: 312785 | ner_oraciones: 486450


In [9]:
# ner_articulos.csv
campos_art = ['url', 'fuente', 'fecha', 'titulo', 'entidad', 'tipo', 'subtipo', 'n_menciones']
with open(OUTPUT_DIR / 'ner_articulos.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=campos_art)
    w.writeheader()
    w.writerows(rows_articulos)
print('Guardado: ner_articulos.csv')

# ner_oraciones.csv
campos_or = ['url', 'fuente', 'fecha', 'titulo', 'oracion_idx', 'oracion', 'entidad', 'tipo', 'subtipo']
with open(OUTPUT_DIR / 'ner_oraciones.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=campos_or)
    w.writeheader()
    w.writerows(rows_oraciones)
print('Guardado: ner_oraciones.csv')

# ner_resumen.csv — ranking global
conteo_global = defaultdict(lambda: {'n_articulos': 0, 'n_menciones': 0})
for row in rows_articulos:
    key = (row['entidad'], row['tipo'], row['subtipo'])
    conteo_global[key]['n_articulos'] += 1
    conteo_global[key]['n_menciones'] += row['n_menciones']

resumen = [
    {'entidad': k[0], 'tipo': k[1], 'subtipo': k[2], **v}
    for k, v in conteo_global.items()
]
resumen.sort(key=lambda x: x['n_menciones'], reverse=True)

with open(OUTPUT_DIR / 'ner_resumen.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['entidad', 'tipo', 'subtipo', 'n_articulos', 'n_menciones'])
    w.writeheader()
    w.writerows(resumen)
print('Guardado: ner_resumen.csv')

Guardado: ner_articulos.csv


Guardado: ner_oraciones.csv
Guardado: ner_resumen.csv


In [10]:
print('=' * 60)
print('RESUMEN DE ENTIDADES DETECTADAS')
print('=' * 60)

for tipo in ['ORG', 'PER', 'LOC', 'IND_ECO', 'MISC']:
    subset = [r for r in resumen if r['tipo'] == tipo]
    if not subset:
        continue
    print(f'\n--- {tipo} (top 10) ---')
    for r in subset[:10]:
        sub = f" [{r['subtipo']}]" if r['subtipo'] else ''
        print(f"  {r['entidad']:40s}  art:{r['n_articulos']:5d}  menciones:{r['n_menciones']:6d}{sub}")

print(f'\nTotal entidades unicas: {len(resumen)}')
print(f'Outputs en: {OUTPUT_DIR.resolve()}')

RESUMEN DE ENTIDADES DETECTADAS

--- ORG (top 10) ---
  bcra                                      art: 2789  menciones:  7607
  banco central                             art: 3367  menciones:  6336
  fmi                                       art: 1555  menciones:  3608
  banco nación                              art: 1577  menciones:  2585
  tna                                       art:  810  menciones:  2124
  arca                                      art:  610  menciones:  1776
  congreso                                  art: 1020  menciones:  1495
  anses                                     art:  405  menciones:  1097
  fed                                       art:  473  menciones:  1020
  carrefour                                 art:  243  menciones:   808

--- PER (top 10) ---
  milei                                     art: 1993  menciones:  4840
  caputo                                    art: 1574  menciones:  3870
  javier milei                              art: 2487  menci

## Cruce con sentimiento FinBETO (opcional)

Llama a `cruce_ner_sentimiento.py`, que une `ner_oraciones.csv` con el sentimiento por oración de FinBETO (`../sentimiento_finbert/beto_oraciones_finetuned.csv`) y genera `ner_sentimiento.csv`, `ner_sentimiento_diario.csv` y `ner_sentimiento_reporte.json`. No usa el baseline sin fine-tuning: su índice de oración no incluye el título y queda corrido en 1 respecto de este NER.

In [11]:
import subprocess, sys

oraciones_ft = Path('../sentimiento_finbert/beto_oraciones_finetuned.csv')
if not oraciones_ft.exists():
    print(f'No encontrado: {oraciones_ft}')
else:
    subprocess.run([sys.executable, 'cruce_ner_sentimiento.py', '--oraciones', str(oraciones_ft)], check=True)